In [1]:
%pip install -q nfl-data-py

import pandas as pd
import numpy as np
import nfl_data_py as nfl
import data_collection as data
nfl_teams = pd.read_csv('nfl_teams.csv')
team_map = dict(zip(nfl_teams['team_name'], nfl_teams['team_id']))


Note: you may need to restart the kernel to use updated packages.


In [2]:
from datetime import datetime

# Set the season to analyze (e.g., 2024). Change if needed.
season = 2025
week = 4

# Load play-by-play data for the chosen season
def get_weekly_scorers(season, week):
    pbp = nfl.import_pbp_data(years=[season])

    # Filter for regular season, Week 1
    week_pbp = pbp[(pbp['week'] == week)]

    # Keep only touchdown plays
    week_tds = week_pbp[week_pbp['touchdown'] == 1]

    # Count TDs per scorer. Prefer id+name if both available, else fall back to name only
    use_cols = [c for c in ['td_player_id', 'td_player_name'] if c in week_tds.columns]
    if use_cols:
        scorers = (
            week_tds.dropna(subset=use_cols)
            .groupby(use_cols)
            .size()
            .reset_index(name='tds')
        )
        if 'td_player_id' in use_cols:
            scorers = scorers.rename(columns={'td_player_name': 'player', 'td_player_id': 'player_id'})
        else:
            scorers = scorers.rename(columns={'td_player_name': 'player'})
    else:
        # Fallback if td_* columns not present; derive from rusher/receiver
        rush = week_tds.dropna(subset=['rusher_player_id'])[['rusher_player_id', 'rusher_player_name']]
        rec = week_tds.dropna(subset=['receiver_player_id'])[['receiver_player_id', 'receiver_player_name']]
        rush.columns = ['player_id', 'player']
        rec.columns = ['player_id', 'player']
        both = pd.concat([rush, rec], ignore_index=True)
        scorers = both.groupby(['player_id', 'player']).size().reset_index(name='tds')

    return scorers

    # Show results
scorers = get_weekly_scorers(2025, week)
scorers.head(10)


2025 done.
Downcasting floats.


,player_id,player,tds
0,00-0031381,D.Adams,1
1,00-0031610,D.Waller,2
2,00-0032464,K.Raymond,1
3,00-0033077,D.Prescott,1
4,00-0033090,H.Henry,1
5,00-0033280,C.McCaffrey,1
6,00-0033555,M.Hollins,1
7,00-0033857,J.Smith-Schuster,1
8,00-0034348,C.Sutton,1
9,00-0034351,D.Goedert,2


In [3]:
predictions = pd.read_csv(f'data/predictions_week_{week}.csv')

#Keep only player_id, player_display_name, predicted_touchdown_probability, and model_edge
predictions = predictions[['player_id', 'player_display_name', 'position','team', 'predicted_touchdown_probability', 'price', 'model_edge','market_implied_prob']]

#Sort by predicted_touchdown_probability in descending order
predictions = predictions.sort_values(by='predicted_touchdown_probability', ascending=False)

#Show the top 50 players
predictions.head(10)



,player_id,player_display_name,position,team,predicted_touchdown_probability,price,model_edge,market_implied_prob
0,00-0036158,J.K. Dobbins,RB,DEN,0.585353,-145.0,-0.006484,0.591837
1,00-0037248,James Cook,RB,BUF,0.574253,-175.0,-0.062110,0.636364
2,00-0037840,Kyren Williams,RB,LA,0.561928,-190.0,-0.093245,0.655172
3,00-0032764,Derrick Henry,RB,BAL,0.549887,-165.0,-0.072754,0.622642
4,00-0039139,Jahmyr Gibbs,RB,DET,0.532050,-170.0,-0.097579,0.629630
5,00-0036223,Jonathan Taylor,RB,IND,0.506200,-205.0,-0.165931,0.672131
6,00-0040784,Quinshon Judkins,RB,CLE,0.492813,145.0,0.084649,0.408163
7,00-0035700,Josh Jacobs,RB,GB,0.481048,-200.0,-0.185618,0.666667
8,00-0040715,Cam Skattebo,RB,NYG,0.479060,-115.0,-0.055824,0.534884
9,00-0040122,Ashton Jeanty,RB,LV,0.467590,-140.0,-0.115743,0.583333


In [4]:
# Join predictions with scorers: prefer player_id, else fall back to name
if 'player_id' in scorers.columns:
    pred_scored = predictions.merge(
        scorers[['player_id', 'tds']], on='player_id', how='inner'
    )
else:
    pred_scored = predictions.merge(
        scorers[['player', 'tds']], left_on='player_display_name', right_on='player', how='inner'
    )

pred_scored.sort_values(['predicted_touchdown_probability'], ascending=[False]).head(10)

,player_id,player_display_name,position,team,predicted_touchdown_probability,price,model_edge,market_implied_prob,tds
0,00-0037248,James Cook,RB,BUF,0.574253,-175.0,-0.062110,0.636364,1
1,00-0039139,Jahmyr Gibbs,RB,DET,0.532050,-170.0,-0.097579,0.629630,1
2,00-0040784,Quinshon Judkins,RB,CLE,0.492813,145.0,0.084649,0.408163,1
3,00-0035700,Josh Jacobs,RB,GB,0.481048,-200.0,-0.185618,0.666667,2
4,00-0040122,Ashton Jeanty,RB,LV,0.467590,-140.0,-0.115743,0.583333,3
5,00-0036973,Travis Etienne,RB,JAX,0.463199,125.0,0.018755,0.444444,1
6,00-0033280,Christian McCaffrey,RB,SF,0.460943,-190.0,-0.194229,0.655172,1
7,00-0036997,Javonte Williams,RB,DAL,0.460366,125.0,0.015921,0.444444,1
8,00-0038542,Bijan Robinson,RB,ATL,0.453948,-180.0,-0.188909,0.642857,1
9,00-0034844,Saquon Barkley,RB,PHI,0.447958,-150.0,-0.152042,0.600000,1


In [5]:
###Simulate betting on the top 10 running backs
def simulate_betting(df, scorers):
    stake = 10.0

    bets = df.copy()

    # Merge to mark hits
    bets = bets.merge(
        scorers[['player_id', 'tds']], on='player_id', how='left'
    )
    bets['tds'] = bets['tds'].fillna(0).astype(int)
    bets['hit'] = bets['tds'] > 0

    # American odds payout logic
    # profit_if_win = stake * (odds/100) if odds > 0 else stake * (100/abs(odds))
    # profit_if_loss = -stake
    is_plus = bets['price'] > 0
    profit_if_win = stake * (bets['price'] / 100.0)
    profit_if_win = profit_if_win.where(is_plus, stake * (100.0 / bets['price'].abs()))

    bets['profit'] = np.where(bets['hit'], profit_if_win, -stake)

    # Add total return
    bets['return'] = stake + bets['profit']

    # Summary metrics
    num_bets = len(bets)
    hits = int(bets['hit'].sum())
    hit_rate = hits / num_bets if num_bets else 0.0
    total_profit = float(bets['profit'].sum())
    roi = total_profit / (stake * num_bets) if num_bets else 0.0

    summary = {
        'bets': num_bets,
        'hits': hits,
        'hit_rate': round(hit_rate, 3),
        'total_profit': round(total_profit, 2),
        'roi': round(roi, 3)
    }

    display(summary)

    # Show detailed results
    cols = [
        'player_id', 'player_display_name', 'team', 'position', 'price',
        'predicted_touchdown_probability', 'model_edge', 'tds', 'hit', 'profit', 'return'
    ]
    return bets[cols].sort_values(['predicted_touchdown_probability'], ascending=[False]).reset_index(drop=True)



In [6]:
# output players from predictions that play for 'PHI', 'KC', 'LAC' or 'DAL'

#find players with model_edge > 0 and price < 500
ev = predictions[predictions['model_edge'] >= 0.07] 
ev = ev[ev['price'] <= 400]

#sort by model_edge in descending order
ev = ev.sort_values('model_edge', ascending=False)
ev







,player_id,player_display_name,position,team,predicted_touchdown_probability,price,model_edge,market_implied_prob
30,00-0033921,Chris Godwin Jr.,WR,TB,0.389796,280.0,0.126639,0.263158
68,00-0040170,Elic Ayomanor,WR,TEN,0.319741,390.0,0.115659,0.204082
23,00-0036912,DeVonta Smith,WR,PHI,0.421029,220.0,0.108529,0.312500
72,00-0036139,Rico Dowdle,RB,CAR,0.311524,370.0,0.098758,0.212766
64,00-0038117,Wan'Dale Robinson,WR,NYG,0.323956,320.0,0.085861,0.238095
6,00-0040784,Quinshon Judkins,RB,CLE,0.492813,145.0,0.084649,0.408163
42,00-0036252,Michael Pittman,WR,IND,0.376602,230.0,0.073572,0.303030


In [7]:
simulate_betting(ev, scorers)

{'bets': 7, 'hits': 2, 'hit_rate': 0.286, 'total_profit': -12.5, 'roi': -0.179}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0040784,Quinshon Judkins,CLE,RB,145.0,0.492813,0.084649,1,True,14.5,24.5
1,00-0036912,DeVonta Smith,PHI,WR,220.0,0.421029,0.108529,0,False,-10.0,0.0
2,00-0033921,Chris Godwin Jr.,TB,WR,280.0,0.389796,0.126639,0,False,-10.0,0.0
3,00-0036252,Michael Pittman,IND,WR,230.0,0.376602,0.073572,1,True,23.0,33.0
4,00-0038117,Wan'Dale Robinson,NYG,WR,320.0,0.323956,0.085861,0,False,-10.0,0.0
5,00-0040170,Elic Ayomanor,TEN,WR,390.0,0.319741,0.115659,0,False,-10.0,0.0
6,00-0036139,Rico Dowdle,CAR,RB,370.0,0.311524,0.098758,0,False,-10.0,0.0


In [8]:
### Get top 10 rb, wr, te, qb from predictions
top_rb = predictions[predictions['position'] == 'RB'].sort_values('predicted_touchdown_probability', ascending=False).head(10)
top_wr = predictions[predictions['position'] == 'WR'].sort_values('predicted_touchdown_probability', ascending=False).head(10)
top_te = predictions[predictions['position'] == 'TE'].sort_values('predicted_touchdown_probability', ascending=False).head(5)
top_qb = predictions[predictions['position'] == 'QB'].sort_values('predicted_touchdown_probability', ascending=False).head(5)

top_rb



,player_id,player_display_name,position,team,predicted_touchdown_probability,price,model_edge,market_implied_prob
0,00-0036158,J.K. Dobbins,RB,DEN,0.585353,-145.0,-0.006484,0.591837
1,00-0037248,James Cook,RB,BUF,0.574253,-175.0,-0.062110,0.636364
2,00-0037840,Kyren Williams,RB,LA,0.561928,-190.0,-0.093245,0.655172
3,00-0032764,Derrick Henry,RB,BAL,0.549887,-165.0,-0.072754,0.622642
4,00-0039139,Jahmyr Gibbs,RB,DET,0.532050,-170.0,-0.097579,0.629630
5,00-0036223,Jonathan Taylor,RB,IND,0.506200,-205.0,-0.165931,0.672131
6,00-0040784,Quinshon Judkins,RB,CLE,0.492813,145.0,0.084649,0.408163
7,00-0035700,Josh Jacobs,RB,GB,0.481048,-200.0,-0.185618,0.666667
8,00-0040715,Cam Skattebo,RB,NYG,0.479060,-115.0,-0.055824,0.534884
9,00-0040122,Ashton Jeanty,RB,LV,0.467590,-140.0,-0.115743,0.583333


In [9]:
simulate_betting(top_rb, scorers)


{'bets': 10, 'hits': 5, 'hit_rate': 0.5, 'total_profit': -11.76, 'roi': -0.118}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0036158,J.K. Dobbins,DEN,RB,-145.0,0.585353,-0.006484,0,False,-10.000000,0.000000
1,00-0037248,James Cook,BUF,RB,-175.0,0.574253,-0.062110,1,True,5.714286,15.714286
2,00-0037840,Kyren Williams,LA,RB,-190.0,0.561928,-0.093245,0,False,-10.000000,0.000000
3,00-0032764,Derrick Henry,BAL,RB,-165.0,0.549887,-0.072754,0,False,-10.000000,0.000000
4,00-0039139,Jahmyr Gibbs,DET,RB,-170.0,0.532050,-0.097579,1,True,5.882353,15.882353
5,00-0036223,Jonathan Taylor,IND,RB,-205.0,0.506200,-0.165931,0,False,-10.000000,0.000000
6,00-0040784,Quinshon Judkins,CLE,RB,145.0,0.492813,0.084649,1,True,14.500000,24.500000
7,00-0035700,Josh Jacobs,GB,RB,-200.0,0.481048,-0.185618,2,True,5.000000,15.000000
8,00-0040715,Cam Skattebo,NYG,RB,-115.0,0.479060,-0.055824,0,False,-10.000000,0.000000
9,00-0040122,Ashton Jeanty,LV,RB,-140.0,0.467590,-0.115743,3,True,7.142857,17.142857


In [10]:
#filter top_wr to only include players with model_edge > 0.05 and price < 500
#top_wr = top_wr[top_wr['model_edge'] > 0.05]
#top_wr = top_wr[top_wr['price'] < 500]
simulate_betting(top_wr, scorers)


{'bets': 10, 'hits': 5, 'hit_rate': 0.5, 'total_profit': 16.5, 'roi': 0.165}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0036963,Amon-Ra St. Brown,DET,WR,125.0,0.439566,-0.004878,2,True,12.5,22.5
1,00-0038544,Quentin Johnston,LAC,WR,170.0,0.429084,0.058714,1,True,17.0,27.0
2,00-0034348,Courtland Sutton,DEN,WR,135.0,0.426042,0.000510,1,True,13.5,23.5
3,00-0039075,Puka Nacua,LA,WR,105.0,0.424026,-0.063779,1,True,10.5,20.5
4,00-0030279,Keenan Allen,LAC,WR,170.0,0.422232,0.051861,0,False,-10.0,0.0
5,00-0036912,DeVonta Smith,PHI,WR,220.0,0.421029,0.108529,0,False,-10.0,0.0
6,00-0033040,Tyreek Hill,MIA,WR,135.0,0.403043,-0.022489,0,False,-10.0,0.0
7,00-0031381,Davante Adams,LA,WR,130.0,0.399786,-0.034997,1,True,13.0,23.0
8,00-0033921,Chris Godwin Jr.,TB,WR,280.0,0.389796,0.126639,0,False,-10.0,0.0
9,00-0038543,Jaxon Smith-Njigba,SEA,WR,150.0,0.389036,-0.010964,0,False,-10.0,0.0


In [11]:
simulate_betting(top_te, scorers)

{'bets': 5, 'hits': 1, 'hit_rate': 0.2, 'total_profit': -22.0, 'roi': -0.44}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0037744,Trey McBride,ARI,TE,165.0,0.368621,-0.008737,0,False,-10.0,0.0
1,00-0039338,Brock Bowers,LV,TE,155.0,0.366157,-0.026000,0,False,-10.0,0.0
2,00-0033090,Hunter Henry,NE,TE,180.0,0.344307,-0.012835,1,True,18.0,28.0
3,00-0038996,Tucker Kraft,GB,TE,200.0,0.311546,-0.021787,0,False,-10.0,0.0
4,00-0034753,Mark Andrews,BAL,TE,235.0,0.301209,0.002701,0,False,-10.0,0.0


In [12]:
simulate_betting(top_qb, scorers)

{'bets': 5, 'hits': 1, 'hit_rate': 0.2, 'total_profit': -33.75, 'roi': -0.675}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0036389,Jalen Hurts,PHI,QB,-135.0,0.377367,-0.197101,0,False,-10.00,0.00
1,00-0034796,Lamar Jackson,BAL,QB,180.0,0.307916,-0.049227,0,False,-10.00,0.00
2,00-0035710,Daniel Jones,IND,QB,165.0,0.300861,-0.076497,0,False,-10.00,0.00
3,00-0039910,Jayden Daniels,WAS,QB,245.0,0.271166,-0.018689,0,False,-10.00,0.00
4,00-0034857,Josh Allen,BUF,QB,-160.0,0.267259,-0.348125,1,True,6.25,16.25


In [13]:
ev_wr = predictions[predictions['model_edge'] >= 0.00]
ev_wr = ev_wr[ev_wr['position'] == 'WR'].sort_values('predicted_touchdown_probability', ascending=False).head(10)
simulate_betting(ev_wr, scorers)


{'bets': 10, 'hits': 3, 'hit_rate': 0.3, 'total_profit': -16.5, 'roi': -0.165}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0038544,Quentin Johnston,LAC,WR,170.0,0.429084,0.058714,1,True,17.0,27.0
1,00-0034348,Courtland Sutton,DEN,WR,135.0,0.426042,0.000510,1,True,13.5,23.5
2,00-0030279,Keenan Allen,LAC,WR,170.0,0.422232,0.051861,0,False,-10.0,0.0
3,00-0036912,DeVonta Smith,PHI,WR,220.0,0.421029,0.108529,0,False,-10.0,0.0
4,00-0033921,Chris Godwin Jr.,TB,WR,280.0,0.389796,0.126639,0,False,-10.0,0.0
5,00-0039916,Ricky Pearsall,SF,WR,185.0,0.384001,0.033123,0,False,-10.0,0.0
6,00-0038563,Tre Tucker,LV,WR,220.0,0.376744,0.064244,0,False,-10.0,0.0
7,00-0036252,Michael Pittman,IND,WR,230.0,0.376602,0.073572,1,True,23.0,33.0
8,00-0039915,Ladd McConkey,LAC,WR,170.0,0.373358,0.002987,0,False,-10.0,0.0
9,00-0036259,Jauan Jennings,SF,WR,170.0,0.370493,0.000122,0,False,-10.0,0.0


In [14]:
top_predictors = predictions[predictions['predicted_touchdown_probability'] >= 0.50]
simulate_betting(top_predictors, scorers)

{'bets': 6, 'hits': 2, 'hit_rate': 0.333, 'total_profit': -28.4, 'roi': -0.473}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0036158,J.K. Dobbins,DEN,RB,-145.0,0.585353,-0.006484,0,False,-10.000000,0.000000
1,00-0037248,James Cook,BUF,RB,-175.0,0.574253,-0.062110,1,True,5.714286,15.714286
2,00-0037840,Kyren Williams,LA,RB,-190.0,0.561928,-0.093245,0,False,-10.000000,0.000000
3,00-0032764,Derrick Henry,BAL,RB,-165.0,0.549887,-0.072754,0,False,-10.000000,0.000000
4,00-0039139,Jahmyr Gibbs,DET,RB,-170.0,0.532050,-0.097579,1,True,5.882353,15.882353
5,00-0036223,Jonathan Taylor,IND,RB,-205.0,0.506200,-0.165931,0,False,-10.000000,0.000000


In [15]:
top_vegas = predictions.sort_values('market_implied_prob', ascending=False).head(19)
simulate_betting(top_vegas, scorers)

{'bets': 19,
 'hits': 10,
 'hit_rate': 0.526,
 'total_profit': -26.93,
 'roi': -0.142}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0036158,J.K. Dobbins,DEN,RB,-145.0,0.585353,-0.006484,0,False,-10.000000,0.000000
1,00-0037248,James Cook,BUF,RB,-175.0,0.574253,-0.062110,1,True,5.714286,15.714286
2,00-0037840,Kyren Williams,LA,RB,-190.0,0.561928,-0.093245,0,False,-10.000000,0.000000
3,00-0032764,Derrick Henry,BAL,RB,-165.0,0.549887,-0.072754,0,False,-10.000000,0.000000
4,00-0039139,Jahmyr Gibbs,DET,RB,-170.0,0.532050,-0.097579,1,True,5.882353,15.882353
5,00-0036223,Jonathan Taylor,IND,RB,-205.0,0.506200,-0.165931,0,False,-10.000000,0.000000
6,00-0035700,Josh Jacobs,GB,RB,-200.0,0.481048,-0.185618,2,True,5.000000,15.000000
7,00-0040715,Cam Skattebo,NYG,RB,-115.0,0.479060,-0.055824,0,False,-10.000000,0.000000
8,00-0040122,Ashton Jeanty,LV,RB,-140.0,0.467590,-0.115743,3,True,7.142857,17.142857
9,00-0033280,Christian McCaffrey,SF,RB,-190.0,0.460943,-0.194229,1,True,5.263158,15.263158


In [16]:
vegas_similar = predictions[predictions['model_edge'] < 0.05]
vegas_similar = vegas_similar[vegas_similar['model_edge'] > 0]
vegas_similar = vegas_similar[vegas_similar['price'] < 300]
simulate_betting(vegas_similar, scorers)

{'bets': 10, 'hits': 3, 'hit_rate': 0.3, 'total_profit': -31.5, 'roi': -0.315}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0036973,Travis Etienne,JAX,RB,125.0,0.463199,0.018755,1,True,12.5,22.5
1,00-0036997,Javonte Williams,DAL,RB,125.0,0.460366,0.015921,1,True,12.5,22.5
2,00-0034348,Courtland Sutton,DEN,WR,135.0,0.426042,0.000510,1,True,13.5,23.5
3,00-0037228,Jaylen Warren,PIT,RB,150.0,0.412550,0.012550,0,False,-10.0,0.0
4,00-0039916,Ricky Pearsall,SF,WR,185.0,0.384001,0.033123,0,False,-10.0,0.0
5,00-0039915,Ladd McConkey,LAC,WR,170.0,0.373358,0.002987,0,False,-10.0,0.0
6,00-0036259,Jauan Jennings,SF,WR,170.0,0.370493,0.000122,0,False,-10.0,0.0
7,00-0034827,DJ Moore,CHI,WR,200.0,0.349453,0.016120,0,False,-10.0,0.0
8,00-0034753,Mark Andrews,BAL,TE,235.0,0.301209,0.002701,0,False,-10.0,0.0
9,00-0036040,Juwan Johnson,NO,TE,265.0,0.296369,0.022396,0,False,-10.0,0.0


In [17]:
top_25 = predictions.head(25)
simulate_betting(top_25, scorers)

{'bets': 25, 'hits': 15, 'hit_rate': 0.6, 'total_profit': 42.92, 'roi': 0.172}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0036158,J.K. Dobbins,DEN,RB,-145.0,0.585353,-0.006484,0,False,-10.000000,0.000000
1,00-0037248,James Cook,BUF,RB,-175.0,0.574253,-0.062110,1,True,5.714286,15.714286
2,00-0037840,Kyren Williams,LA,RB,-190.0,0.561928,-0.093245,0,False,-10.000000,0.000000
3,00-0032764,Derrick Henry,BAL,RB,-165.0,0.549887,-0.072754,0,False,-10.000000,0.000000
4,00-0039139,Jahmyr Gibbs,DET,RB,-170.0,0.532050,-0.097579,1,True,5.882353,15.882353
5,00-0036223,Jonathan Taylor,IND,RB,-205.0,0.506200,-0.165931,0,False,-10.000000,0.000000
6,00-0040784,Quinshon Judkins,CLE,RB,145.0,0.492813,0.084649,1,True,14.500000,24.500000
7,00-0035700,Josh Jacobs,GB,RB,-200.0,0.481048,-0.185618,2,True,5.000000,15.000000
8,00-0040715,Cam Skattebo,NYG,RB,-115.0,0.479060,-0.055824,0,False,-10.000000,0.000000
9,00-0040122,Ashton Jeanty,LV,RB,-140.0,0.467590,-0.115743,3,True,7.142857,17.142857


In [18]:
import pandas as pd
from typing import Optional


def append_week_lines_to_historic(
    week_lines_csv: str = "data/week_4_lines.csv",
    historic_csv: str = "data/historic_lines.csv",
    teams_csv: str = "nfl_teams.csv",
    schedule_season: int = 2025,
    schedule_week: int = 4,
    bookmaker_preference: Optional[str] = "DraftKings"
) -> int:
    """Append weekly spread lines into historic_lines.csv, preserving schema.

    - Reads week-level lines in the format of week_2_lines.csv (two rows/team per game).
    - Maps full team names to IDs matching historic_lines.csv via nfl_teams.csv.
    - Aggregates to one row per game: picks the favorite (negative point), keeps total.
    - Appends new rows to historic_lines.csv with the same columns and blank index header.

    Returns
    -------
    int
        Number of rows appended (deduped against existing season/week/home/away).
    """
    # Load team mapping (full name -> team_id used in historic file)
    teams_df = pd.read_csv(teams_csv)
    name_to_id = dict(zip(teams_df["team_name"].astype(str), teams_df["team_id"].astype(str)))

    # Load week lines and filter to spreads (and bookmaker if provided)
    week_df = pd.read_csv(week_lines_csv)
    if "market" in week_df.columns:
        week_df = week_df[week_df["market"].str.lower() == "spreads"].copy()
    if bookmaker_preference and "bookmaker" in week_df.columns:
        week_df = week_df[week_df["bookmaker"].astype(str) == bookmaker_preference].copy()

    # Normalize numeric fields
    if "point" in week_df.columns:
        week_df["point"] = pd.to_numeric(week_df["point"], errors="coerce")
    # 'over/under' has a slash in the name; keep safe access
    ou_col = "over/under" if "over/under" in week_df.columns else (
        "over_under" if "over_under" in week_df.columns else None
    )
    if ou_col is not None:
        week_df[ou_col] = pd.to_numeric(week_df[ou_col], errors="coerce")

    # Group to one row per game
    required_cols = {"game_id", "home_team", "away_team", "label"}
    missing = [c for c in required_cols if c not in week_df.columns]
    if missing:
        raise ValueError(f"Missing required columns in {week_lines_csv}: {missing}")

    records = []
    for game_id, grp in week_df.groupby("game_id", sort=False):
        home_team_name = str(grp["home_team"].iloc[0])
        away_team_name = str(grp["away_team"].iloc[0])

        # Determine favorite: row with the most negative spread (minimum point)
        grp_nonnull = grp.dropna(subset=["point"]) if "point" in grp.columns else grp.copy()
        if grp_nonnull.empty:
            # If we can't determine a favorite, skip this game
            continue
        fav_idx = grp_nonnull["point"].idxmin()
        fav_team_name = str(grp_nonnull.loc[fav_idx, "label"])  # team name in the bet label
        spread_favorite = float(grp_nonnull.loc[fav_idx, "point"])  # should be negative

        # Over/Under line: take first non-null within the game
        if ou_col is not None:
            ou_series = grp_nonnull[ou_col].dropna()
            over_under_line = float(ou_series.iloc[0]) if not ou_series.empty else None
        else:
            over_under_line = None

        # Map to team IDs used in historic file
        home_id = name_to_id.get(home_team_name)
        away_id = name_to_id.get(away_team_name)
        fav_id = name_to_id.get(fav_team_name)

        if home_id is None or away_id is None or fav_id is None:
            # Try a couple of common aliases
            alias = {
                "LA Rams": "Los Angeles Rams",
                "LA Chargers": "Los Angeles Chargers",
                "LV Raiders": "Las Vegas Raiders",
                "Washington": "Washington Commanders",
            }
            home_id = home_id or name_to_id.get(alias.get(home_team_name, home_team_name))
            away_id = away_id or name_to_id.get(alias.get(away_team_name, away_team_name))
            fav_id = fav_id or name_to_id.get(alias.get(fav_team_name, fav_team_name))

        if home_id is None or away_id is None or fav_id is None:
            raise KeyError(
                f"Missing team_id mapping. home='{home_team_name}'->{home_id}, "
                f"away='{away_team_name}'->{away_id}, favorite='{fav_team_name}'->{fav_id}"
            )

        records.append({
            "schedule_season": int(schedule_season),
            "schedule_week": int(schedule_week),
            "team_home": home_team_name,
            "team_away": away_team_name,
            "team_favorite_id": fav_id,
            "spread_favorite": spread_favorite,
            "over_under_line": over_under_line,
            "schedule_playoff": False,
            "team_home_id": home_id,
            "team_away_id": away_id,
        })

    new_rows_df = pd.DataFrame.from_records(records)
    if new_rows_df.empty:
        return 0

    # Load historic lines with existing index (blank header) and dedupe by key
    historic_df = pd.read_csv(historic_csv, index_col=0, low_memory=False)
    historic_df.index.name = ""  # Ensure blank header on index when saving

    new_rows_df["__key"] = (
        new_rows_df["schedule_season"].astype(str)
        + "|" + new_rows_df["schedule_week"].astype(str)
        + "|" + new_rows_df["team_home"].astype(str)
        + "|" + new_rows_df["team_away"].astype(str)
    )
    hist_keys = set(
        (historic_df["schedule_season"].astype(str)
         + "|" + historic_df["schedule_week"].astype(str)
         + "|" + historic_df["team_home"].astype(str)
         + "|" + historic_df["team_away"].astype(str))
        .values
    )

    new_rows_df = new_rows_df[~new_rows_df["__key"].isin(hist_keys)].drop(columns=["__key"])  # anti-join
    if new_rows_df.empty:
        return 0

    # Assign sequential index values continuing from existing max index
    try:
        start_index = int(pd.to_numeric(pd.Series(historic_df.index)).max())
    except Exception:
        # If index isn't numeric for some reason, fall back to length-1
        start_index = len(historic_df) - 1

    new_index = list(range(start_index + 1, start_index + 1 + len(new_rows_df)))
    new_rows_df.index = new_index
    new_rows_df.index.name = ""  # keep blank index header

    # Concatenate and persist
    out_df = pd.concat([historic_df, new_rows_df], axis=0)
    out_df.index.name = ""
    out_df.to_csv(historic_csv)

    return len(new_rows_df)

# Example usage (uncomment to run):
# appended = append_week_lines_to_historic()
# print(f"Appended {appended} rows to historic_lines.csv")



In [19]:
appended = append_week_lines_to_historic()

In [88]:
week_current = pd.read_csv('data/predictions_week_5.csv')
week_current = week_current[['player_id', 'player_display_name', 'position', 'team', 'opponent_team', 'predicted_touchdown_probability', 'price', 'model_edge', 'market_implied_prob']]
top_rb = week_current[week_current['position'] == 'RB'].sort_values('predicted_touchdown_probability', ascending=False).head(10)
top_wr = week_current[week_current['position'] == 'WR'].sort_values('predicted_touchdown_probability', ascending=False).head(10)
top_te = week_current[week_current['position'] == 'TE'].sort_values('predicted_touchdown_probability', ascending=False).head(5)
top_qb = week_current[week_current['position'] == 'QB'].sort_values('predicted_touchdown_probability', ascending=False).head(5)
ev = week_current[week_current['model_edge'] >= 0.07]
ev = ev[ev['price'] <= 400]
ev = ev.sort_values('model_edge', ascending=False)
high_prob = week_current[week_current['predicted_touchdown_probability'] >= 0.50]








In [89]:
top_rb

,player_id,player_display_name,position,team,opponent_team,predicted_touchdown_probability,price,model_edge,market_implied_prob
0,00-0037248,James Cook,RB,BUF,NE,0.595048,-225.0,-0.097260,0.692308
1,00-0036223,Jonathan Taylor,RB,IND,LV,0.579511,-225.0,-0.112797,0.692308
2,00-0036997,Javonte Williams,RB,DAL,NYJ,0.574056,-125.0,0.018500,0.555556
3,00-0039139,Jahmyr Gibbs,RB,DET,CIN,0.552364,-225.0,-0.139944,0.692308
4,00-0034844,Saquon Barkley,RB,PHI,DEN,0.550835,-180.0,-0.092022,0.642857
5,00-0039040,De'Von Achane,RB,MIA,CAR,0.546759,-130.0,-0.018459,0.565217
6,00-0040666,Omarion Hampton,RB,LAC,WAS,0.533581,-150.0,-0.066419,0.600000
7,00-0036973,Travis Etienne,RB,JAX,KC,0.507430,125.0,0.062986,0.444444
8,00-0038134,Kenneth Walker III,RB,SEA,TB,0.496951,-105.0,-0.015244,0.512195
9,00-0036555,Chuba Hubbard,RB,CAR,MIA,0.486443,-105.0,-0.025752,0.512195


In [90]:
top_wr

,player_id,player_display_name,position,team,opponent_team,predicted_touchdown_probability,price,model_edge,market_implied_prob
13,00-0036963,Amon-Ra St. Brown,WR,DET,CIN,0.443090,-110.0,-0.080719,0.523810
16,00-0037247,George Pickens,WR,DAL,NYJ,0.421465,110.0,-0.054725,0.476190
17,00-0031381,Davante Adams,WR,LA,SF,0.419347,100.0,-0.080653,0.500000
18,00-0039075,Puka Nacua,WR,LA,SF,0.417492,100.0,-0.082508,0.500000
19,00-0038544,Quentin Johnston,WR,LAC,WAS,0.414236,140.0,-0.002431,0.416667
22,00-0036252,Michael Pittman,WR,IND,LV,0.408499,195.0,0.069516,0.338983
23,00-0038543,Jaxon Smith-Njigba,WR,SEA,TB,0.407963,145.0,-0.000200,0.408163
26,00-0039849,Marvin Harrison Jr.,WR,ARI,TEN,0.388229,130.0,-0.046554,0.434783
31,00-0037740,Garrett Wilson,WR,NYJ,DAL,0.376359,135.0,-0.049172,0.425532
32,00-0030279,Keenan Allen,WR,LAC,WAS,0.374483,160.0,-0.010132,0.384615


In [91]:
top_te






,player_id,player_display_name,position,team,opponent_team,predicted_touchdown_probability,price,model_edge,market_implied_prob
21,00-0037744,Trey McBride,TE,ARI,TEN,0.408716,170.0,0.038346,0.370370
27,00-0040128,Tyler Warren,TE,IND,LV,0.388127,160.0,0.003511,0.384615
61,00-0039338,Brock Bowers,TE,LV,IND,0.307096,185.0,-0.043781,0.350877
62,00-0039065,Sam LaPorta,TE,DET,CIN,0.305477,190.0,-0.039351,0.344828
66,00-0034351,Dallas Goedert,TE,PHI,DEN,0.296464,350.0,0.074242,0.222222


In [92]:
top_qb

,player_id,player_display_name,position,team,opponent_team,predicted_touchdown_probability,price,model_edge,market_implied_prob
24,00-0036389,Jalen Hurts,QB,PHI,DEN,0.405946,-150.0,-0.194054,0.600000
59,00-0034857,Josh Allen,QB,BUF,NE,0.310529,-150.0,-0.289471,0.600000
65,00-0039910,Jayden Daniels,QB,WAS,LAC,0.301894,190.0,-0.042934,0.344828
77,00-0036945,Justin Fields,QB,NYJ,DAL,0.263853,125.0,-0.180591,0.444444
96,00-0035228,Kyler Murray,QB,ARI,TEN,0.219745,200.0,-0.113588,0.333333


In [93]:
ev

,player_id,player_display_name,position,team,opponent_team,predicted_touchdown_probability,price,model_edge,market_implied_prob
47,00-0038994,Jordan Addison,WR,MIN,CLE,0.335614,300.0,0.085614,0.250000
66,00-0034351,Dallas Goedert,TE,PHI,DEN,0.296464,350.0,0.074242,0.222222
35,00-0036912,DeVonta Smith,WR,PHI,DEN,0.368083,240.0,0.073966,0.294118


In [94]:
high_prob

,player_id,player_display_name,position,team,opponent_team,predicted_touchdown_probability,price,model_edge,market_implied_prob
0,00-0037248,James Cook,RB,BUF,NE,0.595048,-225.0,-0.097260,0.692308
1,00-0036223,Jonathan Taylor,RB,IND,LV,0.579511,-225.0,-0.112797,0.692308
2,00-0036997,Javonte Williams,RB,DAL,NYJ,0.574056,-125.0,0.018500,0.555556
3,00-0039139,Jahmyr Gibbs,RB,DET,CIN,0.552364,-225.0,-0.139944,0.692308
4,00-0034844,Saquon Barkley,RB,PHI,DEN,0.550835,-180.0,-0.092022,0.642857
5,00-0039040,De'Von Achane,RB,MIA,CAR,0.546759,-130.0,-0.018459,0.565217
6,00-0040666,Omarion Hampton,RB,LAC,WAS,0.533581,-150.0,-0.066419,0.600000
7,00-0036973,Travis Etienne,RB,JAX,KC,0.507430,125.0,0.062986,0.444444


In [95]:
ev_wr = week_current[week_current['model_edge'] >= 0.00]
ev_wr = ev_wr[ev_wr['position'] == 'WR'].sort_values('predicted_touchdown_probability', ascending=False).head(10)
ev_wr

,player_id,player_display_name,position,team,opponent_team,predicted_touchdown_probability,price,model_edge,market_implied_prob
22,00-0036252,Michael Pittman,WR,IND,LV,0.408499,195.0,0.069516,0.338983
35,00-0036912,DeVonta Smith,WR,PHI,DEN,0.368083,240.0,0.073966,0.294118
42,00-0033921,Chris Godwin Jr.,WR,TB,SEA,0.345399,220.0,0.032899,0.312500
46,00-0034960,Jakobi Meyers,WR,LV,IND,0.336726,205.0,0.008858,0.327869
47,00-0038994,Jordan Addison,WR,MIN,CLE,0.335614,300.0,0.085614,0.250000
50,00-0039064,Zay Flowers,WR,BAL,HOU,0.328735,215.0,0.011275,0.317460
54,00-0038117,Wan'Dale Robinson,WR,NYG,NO,0.319132,235.0,0.020625,0.298507
57,00-0037664,Alec Pierce,WR,IND,LV,0.311591,310.0,0.067689,0.243902
75,00-0040170,Elic Ayomanor,WR,TEN,ARI,0.271372,390.0,0.067290,0.204082
84,00-0036550,Rashod Bateman,WR,BAL,HOU,0.245361,390.0,0.041279,0.204082


In [96]:
opp_def = pd.read_csv('data/predictions_week_5.csv')
opp_def = opp_def.groupby('opponent_team')['rushing_tds_allowed_to_RB'].mean()
opp_def = opp_def.sort_values(ascending=False)
opp_def





opponent_team
TEN    1.438464
NYG    1.147448
MIN    1.110843
WAS    1.026243
BAL    1.006525
NO     0.959442
CAR    0.956869
DAL    0.925407
DET    0.872188
LV     0.860473
BUF    0.831837
NYJ    0.800888
KC     0.795109
HOU    0.778499
CIN    0.725385
SF     0.702235
CLE    0.612509
IND    0.574584
ARI    0.567479
NE     0.532934
TB     0.497552
DEN    0.291701
LAC    0.266766
MIA    0.247480
PHI    0.240899
JAX    0.219034
SEA    0.037167
LA     0.028826
Name: rushing_tds_allowed_to_RB, dtype: float64

In [97]:
%pip install -q nflreadpy 

import nflreadpy as nfl

Note: you may need to restart the kernel to use updated packages.


In [98]:
import data_collection as data
import pandas as pd
all_years_to_load = range(2020, 2026)
nfl_teams = pd.read_csv('nfl_teams.csv')
team_map = dict(zip(nfl_teams['team_name'], nfl_teams['team_id']))
nfl_df = data.get_all_historic_data(all_years_to_load, team_map)


In [99]:
nfl_df

,player_id,player_display_name,position,team,season,week,carries,rushing_yards,rushing_tds,receptions,...,game_id,home_team,away_team,weekday,gametime,roof,surface,temp,wind,div_game
0,00-0019596,Tom Brady,QB,TB,2020,1,3,9,1,0,...,2020_01_TB_NO,NO,TB,Sunday,16:25,dome,astroturf,0.0,0.0,1
1,00-0020531,Drew Brees,QB,NO,2020,1,2,0,0,0,...,2020_01_TB_NO,NO,TB,Sunday,16:25,dome,astroturf,0.0,0.0,1
2,00-0022127,Jason Witten,TE,LV,2020,1,0,0,0,1,...,2020_01_LV_CAR,CAR,LV,Sunday,13:00,outdoors,grass,81.0,5.0,0
3,00-0022921,Larry Fitzgerald,WR,ARI,2020,1,0,0,0,4,...,2020_01_ARI_SF,SF,ARI,Sunday,16:25,outdoors,grass,66.0,6.0,1
4,00-0022924,Ben Roethlisberger,QB,PIT,2020,1,3,9,0,0,...,2020_01_PIT_NYG,NYG,PIT,Monday,19:15,outdoors,fieldturf,71.0,5.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30336,00-0039075,Puka Nacua,WR,LA,2025,5,0,0,0,10,...,2025_05_SF_LA,LA,SF,Thursday,20:15,dome,matrixturf,0.0,0.0,1
30337,00-0039363,Isaac Guerendo,RB,SF,2025,5,0,0,0,0,...,2025_05_SF_LA,LA,SF,Thursday,20:15,dome,matrixturf,0.0,0.0,1
30338,00-0039738,Blake Corum,RB,LA,2025,5,1,13,0,0,...,2025_05_SF_LA,LA,SF,Thursday,20:15,dome,matrixturf,0.0,0.0,1
30339,00-0039751,Jordan Whittington,WR,LA,2025,5,0,0,0,2,...,2025_05_SF_LA,LA,SF,Thursday,20:15,dome,matrixturf,0.0,0.0,1


In [100]:
### Subset nfl_df to include player_display_name, week, season opponent_team, explosive_receiving_plays,
subset_cols = [
    'player_display_name',
    'week',
    'season',
    'opponent_team',
    'explosive_receiving_plays',
    'explosive_receiving_plays_allowed',
    'explosive_rushing_plays',
    'explosive_rushing_plays_allowed'
]
nfl_df_subset = nfl_df[subset_cols]
nfl_df_subset


,player_display_name,week,season,opponent_team,explosive_receiving_plays,explosive_receiving_plays_allowed,explosive_rushing_plays,explosive_rushing_plays_allowed
0,Tom Brady,1,2020,NO,0.0,3.0,0.0,1.0
1,Drew Brees,1,2020,TB,0.0,2.0,0.0,0.0
2,Jason Witten,1,2020,CAR,0.0,3.0,0.0,0.0
3,Larry Fitzgerald,1,2020,SF,0.0,1.0,0.0,3.0
4,Ben Roethlisberger,1,2020,NYG,0.0,2.0,0.0,2.0
...,...,...,...,...,...,...,...,...
30336,Puka Nacua,5,2025,SF,0.0,7.0,0.0,1.0
30337,Isaac Guerendo,5,2025,LA,0.0,1.0,0.0,0.0
30338,Blake Corum,5,2025,SF,0.0,7.0,0.0,1.0
30339,Jordan Whittington,5,2025,SF,1.0,7.0,0.0,1.0


In [101]:
df = nfl_df_subset
df[f'avg_explosive_receiving_plays'] = df.groupby('player_display_name')['explosive_receiving_plays'].transform(lambda x: x.ewm(alpha=0.3, min_periods=1).mean())

#filter to week 4 2025
df = df[df['week'] == 4]
df = df[df['season'] == 2025]

##sort by avg_explosive_receiving_plays in descending order
df = df.sort_values('avg_explosive_receiving_plays', ascending=False)
df.head(10)








/var/folders/wz/tj0dvj7d3k79ybsp_r75q1jm0000gn/T/ipykernel_8381/3532692550.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[f'avg_explosive_receiving_plays'] = df.groupby('player_display_name')['explosive_receiving_plays'].transform(lambda x: x.ewm(alpha=0.3, min_periods=1).mean())


,player_display_name,week,season,opponent_team,explosive_receiving_plays,explosive_receiving_plays_allowed,explosive_rushing_plays,explosive_rushing_plays_allowed,avg_explosive_receiving_plays
30218,Puka Nacua,4,2025,IND,2.0,4.0,0.0,0.0,1.718312
30173,Jaxon Smith-Njigba,4,2025,ARI,2.0,4.0,0.0,3.0,1.697049
30129,George Pickens,4,2025,GB,3.0,4.0,0.0,0.0,1.690588
30269,Tetairoa McMillan,4,2025,NE,1.0,2.0,0.0,1.0,1.663245
30174,Quentin Johnston,4,2025,NYG,2.0,2.0,0.0,3.0,1.646158
30261,Ricky Pearsall,4,2025,JAX,2.0,6.0,0.0,0.0,1.548370
30078,Justin Jefferson,4,2025,PIT,2.0,4.0,0.0,0.0,1.442964
30019,Courtland Sutton,4,2025,CIN,2.0,5.0,0.0,0.0,1.348758
29974,Davante Adams,4,2025,IND,1.0,4.0,0.0,0.0,1.255459
30271,Emeka Egbuka,4,2025,PHI,1.0,2.0,0.0,0.0,1.218318


In [102]:
opponent_stats_df = (
        nfl_df_subset[['season', 'week', 'opponent_team', 'explosive_receiving_plays_allowed']]
          .groupby(['season', 'week', 'opponent_team'], as_index=False)['explosive_receiving_plays_allowed']
          .mean()
    )
    # Ensure chronological order within each opponent for lag/EWM




In [103]:
opponent_stats_df.sort_values(by=['opponent_team', 'season', 'week'], inplace=True)
    
opponent_stats_df['avg_explosive_receiving_plays_allowed'] = opponent_stats_df.groupby('opponent_team')['explosive_receiving_plays_allowed'].transform(lambda x: x.ewm(alpha=0.3, min_periods=1).mean())

opponent_stats_df = opponent_stats_df[opponent_stats_df['week'] == 4]
opponent_stats_df = opponent_stats_df[opponent_stats_df['season'] == 2025]

opponent_stats_df.sort_values('avg_explosive_receiving_plays_allowed', ascending=False)


,season,week,opponent_team,explosive_receiving_plays_allowed,avg_explosive_receiving_plays_allowed
2790,2025,4,DAL,4.0,4.040838
2796,2025,4,JAX,6.0,3.594172
2786,2025,4,CAR,5.0,3.477670
2788,2025,4,CIN,5.0,3.426380
2782,2025,4,ARI,4.0,3.340922
2813,2025,4,WAS,4.0,3.327802
2792,2025,4,DET,2.0,3.314406
2806,2025,4,NYJ,2.0,3.270831
2812,2025,4,TEN,4.0,3.238316
2798,2025,4,LA,3.0,3.170564


In [108]:
pbp = nfl.load_pbp(2025)
x = data.get_opponent_receiving_explosive_play_allowed(pbp)
x[x['opponent_team'] == 'TB']


,opponent_team,season,week,explosive_receiving_plays_allowed
19,TB,2025,1,3
56,TB,2025,2,5
68,TB,2025,4,2
90,TB,2025,3,1
